In [1]:
import pandas as pd
import numpy as np
import os
import json
from dotenv import load_dotenv
import requests
import time

In [ ]:
import sys
from pathlib import Path

for candidato in (Path.cwd(), *Path.cwd().resolve().parents):
    if (candidato / "src").is_dir():
        if str(candidato) not in sys.path:
            sys.path.insert(0, str(candidato))
        break

from src.paths import RAIZ, DIR_RAW, DIR_PROCESSED, DIR_REPORTS

PATH_AEMET_CRUDO = DIR_RAW / "aemet_diario_8estaciones_2023_2026.parquet"
PATH_PROCESSED = DIR_PROCESSED / "df_final_zonas_temperaturas_2023_2026.parquet"

In [3]:
# Carga las variables del archivo .env en el entorno del sistema
# Si el archivo .env está en la misma carpeta que este notebook, basta con esto:
# Al usar '../.env', le decimos que salga de 'sandbox' y busque el archivo en la raíz
load_dotenv()
# Si tu .env está en una carpeta superior o una ruta específica, puedes indicarla:
# load_dotenv(dotenv_path="../.env")

# Recuperamos el token de forma segura
TOKEN = os.getenv("API_AEMET")

# Verificación rápida (sin mostrar el token entero por seguridad)
if TOKEN:
    print(f"✅ Token cargado correctamente. Longitud: {len(TOKEN)} caracteres.")
else:
    print("❌ No se pudo encontrar la variable ESIOS_TOKEN. Revisa la ruta del archivo .env.")

✅ Token cargado correctamente. Longitud: 287 caracteres.


Comprobación parquet guardado

In [4]:
# 1. Releer el archivo Parquet recién creado
df_clima_releido = pd.read_parquet(PATH_AEMET_CRUDO, engine="pyarrow")
df_clima_releido.columns


Index(['fecha', 'indicativo', 'nombre', 'provincia', 'altitud', 'tmed', 'prec',
       'tmin', 'horatmin', 'tmax', 'horatmax', 'dir', 'velmedia', 'racha',
       'horaracha', 'sol', 'presMax', 'horaPresMax', 'presMin', 'horaPresMin',
       'hrMedia', 'hrMax', 'horaHrMax', 'hrMin', 'horaHrMin', 'pintMax',
       'horaPIntMax'],
      dtype='str')

In [5]:
# Check 1: Shape
shape_ok = df_clima_releido.shape == (10352, 27)
print(f"1. Shape: {df_clima_releido.shape} [{'OK' if shape_ok else 'FAIL'}]")

# Check 2: Registros por estación
conteo_estaciones = df_clima_releido.groupby('indicativo').size()
estaciones_ok = (conteo_estaciones == 1294).all() and len(conteo_estaciones) == 8
print(f"2. Filas por estación (8 estaciones x 1.294): [{'OK' if estaciones_ok else 'FAIL'}]")
if not estaciones_ok:
    print(conteo_estaciones)

# Check 3: Duplicados por indicativo y fecha
num_duplicados = df_clima_releido.duplicated(['indicativo', 'fecha']).sum()
duplicados_ok = num_duplicados == 0
print(f"3. Duplicados (indicativo, fecha): {num_duplicados} [{'OK' if duplicados_ok else 'FAIL'}]")

# Check 4: Días ausentes por estación frente al rango completo
df_clima_releido['fecha'] = pd.to_datetime(df_clima_releido['fecha'])
rango_esperado = pd.date_range('2023-01-01', '2026-07-17')

ausencias = df_clima_releido.groupby('indicativo')['fecha'].apply(
    lambda s: len(rango_esperado.difference(s))
)
ausencias_ok = (ausencias == 0).all()
print(f"4. Días ausentes por estación: [{'OK' if ausencias_ok else 'FAIL'}]")
if not ausencias_ok:
    print(ausencias[ausencias > 0])

# Resumen global
todo_correcto = shape_ok and estaciones_ok and duplicados_ok and ausencias_ok
print("\n" + ("=" * 40))
print(f"RESULTADO GLOBAL: {'Dataset 100% Validado' if todo_correcto else 'Atención: Hay discrepancias'}")
print("=" * 40)

1. Shape: (10352, 27) [OK]
2. Filas por estación (8 estaciones x 1.294): [OK]
3. Duplicados (indicativo, fecha): 0 [OK]
4. Días ausentes por estación: [OK]

RESULTADO GLOBAL: Dataset 100% Validado


In [6]:
# ---------------------------------------------------------
# CONFIGURACIÓN Y CONSTANTES DERIVADAS
# ---------------------------------------------------------
FECHA_INICIO = "2023-01-01"
FECHA_FIN = "2026-07-17"
NUM_ESTACIONES = 8
COLUMNAS_ESPERADAS = 27

RANGO_ESPERADO = pd.date_range(FECHA_INICIO, FECHA_FIN)
DIAS_ESPERADOS_POR_ESTACION = len(RANGO_ESPERADO)
FILAS_TOTALES_ESPERADAS = DIAS_ESPERADOS_POR_ESTACION * NUM_ESTACIONES

# ---------------------------------------------------------
# CARGA DE DATOS
# ---------------------------------------------------------

df_clima_releido["fecha"] = pd.to_datetime(df_clima_releido["fecha"])

# ---------------------------------------------------------
# VALIDACIÓN DE CALIDAD (Sin assert -> ValueError)
# ---------------------------------------------------------

# 1. Shape exacto
if df_clima_releido.shape != (FILAS_TOTALES_ESPERADAS, COLUMNAS_ESPERADAS):
    raise ValueError(
        f"Shape incorrecto: {df_clima_releido.shape}. Se esperaba ({FILAS_TOTALES_ESPERADAS}, {COLUMNAS_ESPERADAS})."
    )

# 2. Número de estaciones y número de días por estación
estaciones_counts = df_clima_releido.groupby("indicativo").size()
if len(estaciones_counts) != NUM_ESTACIONES:
    raise ValueError(
        f"Se esperaban {NUM_ESTACIONES} estaciones pero se encontraron {len(estaciones_counts)}."
    )

if not (estaciones_counts == DIAS_ESPERADOS_POR_ESTACION).all():
    descuadres = estaciones_counts[estaciones_counts != DIAS_ESPERADOS_POR_ESTACION]
    raise ValueError(f"Estaciones con registros descuadrados:\n{descuadres}")

# 3. Duplicados en la clave primaria
num_duplicados = df_clima_releido.duplicated(["indicativo", "fecha"]).sum()
if num_duplicados > 0:
    raise ValueError(f"Se encontraron {num_duplicados} registros duplicados para [indicativo, fecha].")

# 4. Cobertura del rango de fechas
dias_faltantes = df_clima_releido.groupby("indicativo")["fecha"].apply(
    lambda s: len(RANGO_ESPERADO.difference(s))
)
if (dias_faltantes > 0).any():
    raise ValueError(f"Hay días ausentes en el calendario:\n{dias_faltantes[dias_faltantes > 0]}")

print(f"Validation successful: {len(df_clima_releido)} filas cargadas correctamente.")

Validation successful: 10352 filas cargadas correctamente.


In [ ]:
from src.quality import obtener_longitud_maxima_racha


# Aseguramos el orden cronológico estricto por estación para calcular rachas correctamente
df_sorted = df_clima_releido.sort_values(["indicativo", "fecha"]).copy()

resultados_estaciones = []

for indicativo, df_est in df_sorted.groupby("indicativo"):
    # Máscaras booleanas de NaNs
    nan_tmin = df_est["tmin"].isna()
    nan_tmax = df_est["tmax"].isna()
    nan_ambos = nan_tmin & nan_tmax
    nan_cualquiera = nan_tmin | nan_tmax

    # 1. Fechas con NaN (ordenadas)
    fechas_nan_tmin = df_est.loc[nan_tmin, "fecha"].dt.strftime("%Y-%m-%d").tolist()
    fechas_nan_tmax = df_est.loc[nan_tmax, "fecha"].dt.strftime("%Y-%m-%d").tolist()

    # 2. Rachas máximas de días consecutivos con NaN
    racha_max_tmin = obtener_longitud_maxima_racha(nan_tmin)
    racha_max_tmax = obtener_longitud_maxima_racha(nan_tmax)

    # 3. Diagnóstico de coincidencia tmin vs tmax
    total_nan_tmin = nan_tmin.sum()
    total_nan_tmax = nan_tmax.sum()
    dias_estacion_caida = nan_ambos.sum()
    dias_solo_tmin = (nan_tmin & ~nan_tmax).sum()
    dias_solo_tmax = (nan_tmax & ~nan_tmin).sum()

    coincidencia_perfecta = (nan_tmin == nan_tmax).all()

    resultados_estaciones.append({
        "indicativo": indicativo,
        "nans_tmin": total_nan_tmin,
        "nans_tmax": total_nan_tmax,
        "racha_max_tmin": racha_max_tmin,
        "racha_max_tmax": racha_max_tmax,
        "dias_estacion_caida": dias_estacion_caida,
        "dias_solo_tmin": dias_solo_tmin,
        "dias_solo_tmax": dias_solo_tmax,
        "coinciden_exacto": coincidencia_perfecta,
        "fechas_tmin": fechas_nan_tmin,
        "fechas_tmax": fechas_nan_tmax,
    })

# Convertimos a DataFrame de resumen para inspección rápida
df_resumen = pd.DataFrame(resultados_estaciones)

# ---------------------------------------------------------
# MOSTRAR RESULTADOS
# ---------------------------------------------------------
print("=== RESUMEN POR ESTACIÓN ===")
columnas_resumen = [
    "indicativo",
    "nans_tmin",
    "nans_tmax",
    "racha_max_tmin",
    "racha_max_tmax",
    "dias_estacion_caida",
    "dias_solo_tmin",
    "dias_solo_tmax",
    "coinciden_exacto",
]
print(df_resumen[columnas_resumen].to_string(index=False))

print("\n=== DETALLE DE FECHAS CON NaN ===")
for res in resultados_estaciones:
    if res["nans_tmin"] > 0 or res["nans_tmax"] > 0:
        print(f"\nEstación {res['indicativo']}:")
        if res["fechas_tmin"]:
            print(f"  - Fechas NaN tmin ({len(res['fechas_tmin'])}): {res['fechas_tmin']}")
        if res["fechas_tmax"]:
            print(f"  - Fechas NaN tmax ({len(res['fechas_tmax'])}): {res['fechas_tmax']}")
    else:
        print(f"Estación {res['indicativo']}: Sin NaNs en tmin ni tmax.")

In [8]:
df_clima_releido.head()

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,...,horaPresMax,presMin,horaPresMin,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin,pintMax,horaPIntMax
0,2023-01-01,3129,MADRID AEROPUERTO,MADRID,609,10.2,"0,1",3.9,00:23,16.4,...,00,"952,6",19,66,95,04:59,34,14:53,"0,0",<NA>
1,2023-01-02,3129,MADRID AEROPUERTO,MADRID,609,10.6,"0,0",8.4,23:54,12.9,...,Varias,"953,4",02,84,92,22:08,56,00:00,"0,6",02:32
2,2023-01-03,3129,MADRID AEROPUERTO,MADRID,609,8.4,"0,0",2.6,23:59,14.2,...,Varias,"959,0",00,77,94,23:09,52,15:18,"0,0",<NA>
3,2023-01-04,3129,MADRID AEROPUERTO,MADRID,609,6.6,"0,0",-0.8,06:44,14.0,...,Varias,"963,7",17,77,98,08:05,50,15:23,"0,0",<NA>
4,2023-01-05,3129,MADRID AEROPUERTO,MADRID,609,6.5,"0,0",-1.0,Varias,14.0,...,00,"958,3",18,73,98,05:47,46,15:31,"0,0",<NA>


Interpolar temperaturas

In [ ]:
from src.quality import contar_nans_por_estacion
from src.clima import interpolar_temperaturas


In [10]:
df_t_interpolado = interpolar_temperaturas(df_clima_releido)
df_t_interpolado

=== CONTROL DE IMPUTACIÓN DE NaNs ===

[TMIN] NaNs totales: 33 → 1
            Antes  Después
indicativo                
0076           15        0
2539            3        0
3129            1        0
5783           10        1
6155A           3        0
8414A           1        0

[TMAX] NaNs totales: 33 → 1
            Antes  Después
indicativo                
0076           16        0
2539            2        0
3129            2        0
5783           10        1
6155A           2        0
8414A           1        0


,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,...,horaPresMax,presMin,horaPresMin,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin,pintMax,horaPIntMax
0,2023-01-01,0076,BARCELONA AEROPUERTO,BARCELONA,4,12.2,"0,0",8.6,07:22,15.9,...,Varias,"1023,9",15,82,96,Varias,63,10:35,"0,0",<NA>
1,2023-01-02,0076,BARCELONA AEROPUERTO,BARCELONA,4,11.0,"0,0",7.0,06:48,15.0,...,Varias,"1022,8",05,61,86,00:00,47,13:54,"0,0",<NA>
2,2023-01-03,0076,BARCELONA AEROPUERTO,BARCELONA,4,12.9,"0,0",8.2,23:30,17.6,...,Varias,"1025,9",01,77,89,18:00,60,12:20,"0,0",<NA>
3,2023-01-04,0076,BARCELONA AEROPUERTO,BARCELONA,4,11.8,"0,0",7.5,07:08,16.1,...,Varias,"1031,2",Varias,76,89,Varias,57,12:47,"0,0",<NA>
4,2023-01-05,0076,BARCELONA AEROPUERTO,BARCELONA,4,10.6,"0,0",5.7,07:35,15.6,...,00,"1024,8",24,74,82,18:14,49,12:36,"0,0",<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10347,2026-07-13,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249,31.1,"0,0",24.5,04:40,37.7,...,Varias,"979,9",18,37,70,01:20,23,16:30,"0,0",<NA>
10348,2026-07-14,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249,32.0,"0,0",24.3,05:20,39.7,...,08,"982,0",17,41,69,23:50,25,15:10,"0,0",<NA>
10349,2026-07-15,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249,33.0,"0,0",24.2,05:10,41.9,...,07,"983,5",17,35,83,03:40,9,17:00,"0,0",<NA>
10350,2026-07-16,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249,31.2,"0,0",24.2,06:10,38.2,...,07,"981,1",17,42,64,06:10,21,18:20,"0,0",<NA>


Zonas y pesos

In [ ]:
df_zonas_pesos = df_t_interpolado.copy()

from src.clima import ZONAS, PESOS

# ---------------------------------------------------------
# INVARIANTES DE CONFIGURACIÓN ("Let it scream")
# ---------------------------------------------------------

# 1. Los pesos deben sumar 1 exacto (previene discrepancias entre revisiones del padrón)
assert (
    abs(sum(PESOS.values()) - 1.0) < 1e-9
), f"La suma de los pesos debe ser 1.0, pero suma {sum(PESOS.values())}"

# 2. Alineación 1:1 estricta entre el dataset, las zonas y los pesos (caza claves huérfanas/sobrantes)
estaciones_df = set(df_zonas_pesos["indicativo"].unique())
assert (
    estaciones_df == set(ZONAS.keys()) == set(PESOS.keys())
), f"Desalineación de indicativos.\nDF: {estaciones_df}\nZONAS: {set(ZONAS.keys())}\nPESOS: {set(PESOS.keys())}"


# ---------------------------------------------------------
# ASIGNACIÓN DIRECTA SOBRE EL MISMO DATAFRAME
# ---------------------------------------------------------
df_zonas_pesos["zona"] = df_zonas_pesos["indicativo"].map(ZONAS)
df_zonas_pesos["peso"] = df_zonas_pesos["indicativo"].map(PESOS)

# 3. Verificación de completitud tras el mapeo
if df_zonas_pesos["zona"].isna().any():
    raise ValueError("Se encontraron filas con 'zona' no asignada (NaN).")

if df_zonas_pesos["peso"].isna().any():
    raise ValueError("Se encontraron filas con 'peso' no asignado (NaN).")

In [ ]:
# ---------------------------------------------------------
# FUNCIÓN AUXILIAR DE MEDIA PONDERADA
# ---------------------------------------------------------
from src.clima import media_ponderada


# =========================================================
# 1. EJECUCIÓN DEL PASO 3 EN EL DATASET COMPLETO
# =========================================================

# Agrupación por fecha y zona para calcular tmin y tmax ponderadas
df_agrupado = (
    df_zonas_pesos.groupby(["fecha", "zona"])
    .apply(
        lambda g: pd.Series(
            {
                "tmin": media_ponderada(g, "tmin"),
                "tmax": media_ponderada(g, "tmax"),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

# Pivotar para desplegar zonas en columnas
df_pivot = df_agrupado.pivot(
    index="fecha", columns="zona", values=["tmin", "tmax"]
)

# Colapsar el MultiIndex de columnas a f"{variable}_{zona}"
df_pivot.columns = [f"{var}_{zona}" for var, zona in df_pivot.columns]
df_final = df_pivot.reset_index()


# =========================================================
# 2. VERIFICACIÓN Y AUDITORÍA DE SALIDA
# =========================================================

# A. Cuatro números de control global
assert (
    df_final.shape == (1294, 9)
), f"Shape inesperado: {df_final.shape}. Esperado (1294, 9)."
assert df_final["fecha"].is_unique, "La columna 'fecha' contiene duplicados."

nans_totales = df_final.drop(columns=["fecha"]).isna().sum().sum()
assert (
    nans_totales == 2
), f"Se esperaban exactamente 2 NaNs en total, pero se encontraron {nans_totales}."

# Verificar ubicación exacta de los 2 NaNs
nans_guadalquivir_fecha = (
    df_final.loc[
        df_final["tmin_guadalquivir"].isna()
        & df_final["tmax_guadalquivir"].isna(),
        "fecha",
    ]
    .dt.strftime("%Y-%m-%d")
    .tolist()
)

assert nans_guadalquivir_fecha == [
    "2023-12-15"
], f"Ubicación inesperada de NaNs. Se esperaban en '2023-12-15', hallados en: {nans_guadalquivir_fecha}"


# B. Verificaciones de sonda (Controles de consistencia física e instrumental)

# 1. Zona de 1 estación == Serie cruda (Validación de longitud estricta + diferencia cero)
df_sevilla = df_zonas_pesos[df_zonas_pesos["indicativo"] == "5783"].set_index(
    "fecha"
)
df_bilbao = df_zonas_pesos[df_zonas_pesos["indicativo"] == "1082"].set_index(
    "fecha"
)

df_final_indexed = df_final.set_index("fecha")

for var in ["tmin", "tmax"]:
    # Guadalquivir (1293 filas válidas por el NaN del 2023-12-15)
    diff_g = (df_final_indexed[f"{var}_guadalquivir"] - df_sevilla[var]).dropna()
    assert (
        len(diff_g) == 1293
    ), f"Longitud incorrecta en serie diff Guadalquivir ({var}): {len(diff_g)}"
    assert (
        diff_g.abs() < 1e-9
    ).all(), f"Discrepancia detectada en Guadalquivir ({var}) respecto a Sevilla."

    # Cantábrico (1294 filas válidas completas)
    diff_c = (df_final_indexed[f"{var}_cantabrico"] - df_bilbao[var]).dropna()
    assert (
        len(diff_c) == 1294
    ), f"Longitud incorrecta en serie diff Cantábrico ({var}): {len(diff_c)}"
    assert (
        diff_c.abs() < 1e-9
    ).all(), f"Discrepancia detectada en Cantábrico ({var}) respecto a Bilbao."


# 2. Regresión contra la constante validada a mano (Sonda 2023-01-01 en Continental)
fecha_sonda = pd.Timestamp("2023-01-01")
val_tmin_cont_sonda = df_final_indexed.loc[fecha_sonda, "tmin_continental"]
VALOR_ESPERADO_SONDA = 4.302910798122066  # (Madrid*0.401 + Zaragoza*0.081 + Valladolid*0.035) / 0.517

assert (
    abs(val_tmin_cont_sonda - VALOR_ESPERADO_SONDA) < 1e-4 # log 17/7, 4 decimales
), f"Fallo de regresión en tmin continental (2023-01-01): obt. {val_tmin_cont_sonda}, esp. {VALOR_ESPERADO_SONDA}"


# 3. Prueba de estrés de estación faltante con reescalado de pesos (Simulación puntual)
df_simulado = df_zonas_pesos.copy()
# Eliminamos Valladolid ('2539') el 2023-01-01
mask_borrar = (df_simulado["fecha"] == fecha_sonda) & (
    df_simulado["indicativo"] == "2539"
)
df_simulado.loc[mask_borrar, "tmin"] = np.nan

grupo_sim = df_simulado[
    (df_simulado["fecha"] == fecha_sonda)
    & (df_simulado["zona"] == "continental")
]
tmin_sim_calc = media_ponderada(grupo_sim, "tmin")

# Denominador debe ser solo Madrid (0.401) + Zaragoza (0.081) = 0.482
pesos_restantes = PESOS["3129"] + PESOS["9434"]
tmin_manual_sim = (
    df_simulado.loc[
        (df_simulado["fecha"] == fecha_sonda)
        & (df_simulado["indicativo"] == "3129"),
        "tmin",
    ].values[0]
    * PESOS["3129"]
    + df_simulado.loc[
        (df_simulado["fecha"] == fecha_sonda)
        & (df_simulado["indicativo"] == "9434"),
        "tmin",
    ].values[0]
    * PESOS["9434"]
) / pesos_restantes

assert (
    abs(tmin_sim_calc - tmin_manual_sim) < 1e-9
), "El reescalado dinámico de pesos falló al omitir una estación."

print("=== CONTROLES DE AUDITORÍA COMPLETADOS ===")
print("✔ Shape exacto (1.294, 9) y 'fecha' única.")
print("✔ Únicos NaNs: tmin y tmax de guadalquivir en 2023-12-15.")
print(
    "✔ Sondajes de cálculo tmin/tmax validados (G: 1293 / C: 1294 coincidencias)."
)
print(f"✔ Regresión 4 dec contra sonda histórica (tmin = {val_tmin_cont_sonda:.4f}).")
print("✔ Reescalado dinámico de pesos ante estación faltante comprobado.")


# =========================================================
# 3. PERSISTENCIA Y VERIFICACIÓN ROUND-TRIP
# =========================================================

# DIR_PROCESSED viene ya del bootstrap (src.paths); antes se reasignaba
# aquí mismo a la cadena relativa "data/processed", que cuelga de cwd y
# escribía fuera del data/processed/ real (§1.5/§1.7).
PATH_PROCESSED = DIR_PROCESSED / "df_final_zonas_temperaturas_2023_2026.parquet"

# Garantizar existencia de directorio (ya la garantiza src.paths, pero no está de más)
DIR_PROCESSED.mkdir(parents=True, exist_ok=True)

# Guardar artefacto
df_final.to_parquet(PATH_PROCESSED, engine="pyarrow", index=False)

# Releer para round-trip test
df_releido = pd.read_parquet(PATH_PROCESSED, engine="pyarrow")

# Validaciones post-persistencia
assert (
    df_releido.shape == df_final.shape
), f"Fallo en round-trip: Shape {df_releido.shape} != {df_final.shape}"

cols_valor = [col for col in df_releido.columns if col != "fecha"]
for col in cols_valor:
    assert pd.api.types.is_float_dtype(
        df_releido[col]
    ), f"La columna {col} perdió el tipo float tras el round-trip."

assert df_releido.equals(
    df_final
), "El DataFrame releído difiere exactamente del DataFrame original guardado."

print(f"\n[Artefacto exportado exitosamente a '{PATH_PROCESSED}']")